# 01 - Unobserved Components Model (UCM) — Brazil GDP

## Introduction

The **Unobserved Components Model** (UCM) decomposes an observed time series into
latent (unobserved) structural components:

$$
y_t = \mu_t + \psi_t + \varepsilon_t
$$

where:
- $\mu_t$ = **level + trend** (local linear trend)
- $\psi_t$ = **stochastic cycle** (business cycle component)
- $\varepsilon_t$ = **irregular** component (observation noise)

### Level + Trend (Local Linear Trend)
$$
\begin{align}
\mu_t &= \mu_{t-1} + \nu_{t-1} + \eta_t, \quad \eta_t \sim N(0, \sigma^2_\eta) \\
\nu_t &= \nu_{t-1} + \zeta_t, \quad \zeta_t \sim N(0, \sigma^2_\zeta)
\end{align}
$$

### Stochastic Cycle
$$
\begin{bmatrix} \psi_t \\ \psi^*_t \end{bmatrix} =
\rho \begin{bmatrix} \cos \lambda_c & \sin \lambda_c \\ -\sin \lambda_c & \cos \lambda_c \end{bmatrix}
\begin{bmatrix} \psi_{t-1} \\ \psi^*_{t-1} \end{bmatrix} +
\begin{bmatrix} \kappa_t \\ \kappa^*_t \end{bmatrix}
$$

with $0 < \rho < 1$ (damping factor) and $\lambda_c = 2\pi / T$ (cycle frequency,
where $T$ is the period in the same units as the data).

The UCM generalizes the Basic Structural Model (BSM) by adding a stochastic cycle,
making it ideal for extracting **business cycle** dynamics from macroeconomic data.

We apply this model to **Brazil quarterly GDP** (1996–2023) to extract the
trend-cycle decomposition and estimate the periodicity of the Brazilian business cycle.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize

from kalmanbox import UnobservedComponents
from kalmanbox.datasets import load_dataset
from kalmanbox.filters.kalman import KalmanFilter
from kalmanbox.smoothers.rts import RTSSmoother

import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Imports OK')

In [ ]:
# Load Brazil GDP data
df = load_dataset('brazil_pib')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Period: {df["date"].iloc[0]} to {df["date"].iloc[-1]}')
print(f'\nDescriptive statistics:\n{df["pib"].describe()}')
df.head()

In [ ]:
# Exploratory visualization
dates = pd.to_datetime(df['date'])
y_raw = df['pib'].to_numpy(dtype=np.float64)
y = np.log(y_raw)  # Work with log GDP for multiplicative decomposition

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# GDP index
axes[0].plot(dates, y_raw, 'k-', linewidth=0.8)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('GDP Index')
axes[0].set_title('Brazil Quarterly GDP Index (1996-2023)')

# Log GDP
axes[1].plot(dates, y, 'b-', linewidth=0.8)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Log GDP Index')
axes[1].set_title('Log GDP (used for modeling)')

# QoQ growth rate
growth = np.diff(y) * 100
axes[2].bar(dates[1:], growth, width=80, color=['green' if g > 0 else 'red' for g in growth], alpha=0.7)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_xlabel('Date')
axes[2].set_ylabel('QoQ Growth (%)')
axes[2].set_title('Quarter-over-Quarter Growth Rate')

plt.tight_layout()
plt.show()

print(f'Number of observations: {len(y)}')
print(f'Log GDP range: [{y.min():.4f}, {y.max():.4f}]')
print(f'Average QoQ growth: {growth.mean():.3f}%')

## UCM Specification with kalmanbox

We specify a UCM with:
- **Level**: stochastic random walk
- **Trend**: stochastic (local linear trend)
- **Cycle**: stochastic damped cycle (for business cycle extraction)

The cycle frequency $\lambda_c$ needs to be in the business-cycle range.
For quarterly GDP, business cycles typically have periods of **4 to 12 years**
(16 to 48 quarters), so $\lambda_c \in (2\pi/48, \, 2\pi/16) \approx (0.13, \, 0.39)$.

We use a **profile likelihood** approach: evaluate the log-likelihood over a grid
of cycle frequencies, then refine with numerical optimization from the best
starting point. This is a standard technique for UCMs where the cycle frequency
surface may have local optima (Harvey, 1989).

In [ ]:
# UCM specification: level + stochastic trend + stochastic cycle
ucm = UnobservedComponents(
    y,
    level=True,
    trend='stochastic',
    cycle=True,
)

print(f'Model: Unobserved Components')
print(f'Number of observations: {len(y)}')
print(f'Number of states: {ucm._k_states}')
print(f'Parameters: {ucm.param_names}')
print(f'State layout: {ucm._layout}')

In [ ]:
# Profile likelihood over cycle frequency (lambda_c) for visualization
# For each candidate period, evaluate log-likelihood at reasonable parameter values
periods_grid = np.arange(14, 49, 2)
lambda_grid = 2 * np.pi / periods_grid

profile_ll = []
for lam_c in lambda_grid:
    best_ll = -np.inf
    for rho_try in [0.2, 0.4, 0.6, 0.8, 0.95]:
        for s_level in [1e-5, 5e-5, 1e-4, 5e-4]:
            for s_cycle in [5e-5, 1e-4, 5e-4, 1e-3]:
                sp = np.array([1e-7, s_level, 1e-9, rho_try, lam_c, s_cycle])
                try:
                    ll = ucm.loglike(sp)
                    if ll > best_ll:
                        best_ll = ll
                except Exception:
                    pass
    profile_ll.append(best_ll)

profile_ll = np.array(profile_ll)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(periods_grid / 4, profile_ll, 'bo-', markersize=6)
best_idx = np.argmax(profile_ll)
ax.axvline(periods_grid[best_idx] / 4, color='red', linestyle='--', alpha=0.7,
           label=f'Best grid: {periods_grid[best_idx]/4:.1f} years')
ax.set_xlabel('Cycle Period (years)')
ax.set_ylabel('Profile Log-Likelihood')
ax.set_title('Profile Likelihood over Cycle Frequency')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Best period from grid: {periods_grid[best_idx]} quarters ({periods_grid[best_idx]/4:.1f} years)')
print(f'Best profile LL: {profile_ll[best_idx]:.2f}')

In [ ]:
# MLE estimation using global optimizer with business-cycle frequency bounds
# We use differential evolution (gradient-free global optimizer) to avoid
# local optima in the cycle frequency. Bounds constrain the cycle period
# to 14-42 quarters (3.5-10.5 years), the standard business-cycle range.

# Define bounds in unconstrained parameter space
# Params: [sigma2_obs, sigma2_level, sigma2_trend, rho, lambda_c, sigma2_cycle]
bounds_de = [
    (-20, -5),    # log(sigma2_obs)
    (-15, -3),    # log(sigma2_level)
    (-20, -5),    # log(sigma2_trend)
    (-1.5, 3.0),  # arctanh(2*rho-1) -> rho in ~(0.1, 0.995)
    (-3.0, -1.2), # logit(lambda_c/pi) -> period ~14-42 quarters
    (-15, -3),    # log(sigma2_cycle)
]

def neg_loglike(unc):
    try:
        constrained = ucm.transform_params(unc)
        return -ucm.loglike(constrained)
    except Exception:
        return 1e10

de_result = optimize.differential_evolution(
    neg_loglike, bounds_de, seed=42, maxiter=300, tol=1e-10,
    popsize=20, mutation=(0.5, 1.5), recombination=0.9
)

# Refine with L-BFGS-B from DE solution
final_result = optimize.minimize(
    neg_loglike, de_result.x, method='L-BFGS-B', bounds=bounds_de,
    options={'maxiter': 2000, 'ftol': 1e-12}
)
if final_result.fun > de_result.fun:
    final_result = de_result

optimal_params = ucm.transform_params(final_result.x)

print('Estimated UCM parameters (MLE):')
print('=' * 50)
for name, val in zip(ucm.param_names, optimal_params):
    print(f'  {name:20s} = {val:.8f}')
print(f'\nLog-likelihood: {-final_result.fun:.4f}')

lambda_c_hat = optimal_params[ucm.param_names.index('lambda_c')]
rho_hat = optimal_params[ucm.param_names.index('rho')]
period_hat = 2 * np.pi / lambda_c_hat
print(f'\nCycle period: {period_hat:.1f} quarters = {period_hat/4:.1f} years')
print(f'Cycle damping (rho): {rho_hat:.4f}')
if rho_hat > 0 and rho_hat < 1:
    half_life = -np.log(2) / np.log(rho_hat)
    print(f'Cycle half-life: {half_life:.1f} quarters')

In [ ]:
# Decomposition: extract smoothed components
ssm = ucm._build_ssm(optimal_params)
kf = KalmanFilter()
smoother = RTSSmoother()
filter_output = kf.filter(y.reshape(-1, 1), ssm)
smoother_output = smoother.smooth(filter_output, ssm)

layout = ucm._layout
smoothed = smoother_output.smoothed_state

# Extract components
level = smoothed[:, layout['level']['start']]
trend_slope = smoothed[:, layout['trend']['start']]
cycle = smoothed[:, layout['cycle']['start']]
irregular = y - level - cycle  # residual irregular component

# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# Observed + Level (trend-cycle)
axes[0].plot(dates, y, 'k-', linewidth=0.7, alpha=0.6, label='Observed (log GDP)')
axes[0].plot(dates, level, 'b-', linewidth=1.5, label='Level (smoothed)')
axes[0].set_ylabel('Log GDP')
axes[0].set_title('UCM Decomposition — Brazil Quarterly GDP')
axes[0].legend(loc='upper left')

# Trend slope
axes[1].plot(dates, trend_slope * 100, 'g-', linewidth=1.2)
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].set_ylabel('Trend Slope (%)')
axes[1].set_title('Stochastic Trend (Quarterly Growth Rate)')

# Cycle
axes[2].plot(dates, cycle * 100, 'r-', linewidth=1.2)
axes[2].axhline(0, color='gray', linewidth=0.5)
axes[2].fill_between(dates, cycle * 100, 0, alpha=0.15, color='red')
axes[2].set_ylabel('Cycle (%)')
axes[2].set_title(f'Business Cycle (Period ≈ {period_hat/4:.1f} years, ρ = {rho_hat:.3f})')

# Irregular
axes[3].plot(dates, irregular * 100, 'gray', linewidth=0.7)
axes[3].axhline(0, color='gray', linewidth=0.5)
axes[3].set_ylabel('Irregular (%)')
axes[3].set_title('Irregular Component')
axes[3].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Cycle analysis: periodicity, damping, amplitude
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cycle + data overlay
ax1 = axes[0]
y_detrended = y - level  # detrended series
ax1.plot(dates, y_detrended * 100, 'k-', linewidth=0.7, alpha=0.5, label='Detrended GDP')
ax1.plot(dates, cycle * 100, 'r-', linewidth=1.5, label='Extracted Cycle')
ax1.axhline(0, color='gray', linewidth=0.5)
ax1.set_xlabel('Date')
ax1.set_ylabel('Deviation from Trend (%)')
ax1.set_title('Cycle vs Detrended GDP')
ax1.legend()

# Cycle amplitude over time (skip first 10 obs for burn-in from diffuse init)
cycle_star = smoothed[:, layout['cycle']['start'] + 1]
amplitude = np.sqrt(cycle**2 + cycle_star**2) * 100
burn_in = 10
ax2 = axes[1]
ax2.plot(dates[burn_in:], amplitude[burn_in:], 'm-', linewidth=1.2)
ax2.set_xlabel('Date')
ax2.set_ylabel('Amplitude (%)')
ax2.set_title('Cycle Amplitude Over Time (after burn-in)')

plt.tight_layout()
plt.show()

print(f'Cycle Analysis:')
print(f'  Estimated period: {period_hat:.1f} quarters = {period_hat/4:.1f} years')
print(f'  Damping factor (rho): {rho_hat:.4f}')
if rho_hat > 0 and rho_hat < 1:
    half_life = -np.log(2) / np.log(rho_hat)
    print(f'  Half-life: {half_life:.1f} quarters = {half_life/4:.1f} years')
print(f'  Max amplitude (after burn-in): {amplitude[burn_in:].max():.2f}%')
print(f'  Mean amplitude (after burn-in): {amplitude[burn_in:].mean():.2f}%')

In [ ]:
# Forecast 8 quarters ahead
n_forecast = 8

# Get last smoothed state and covariance
a_T = smoother_output.smoothed_state[-1]
P_T = smoother_output.smoothed_cov[-1]

T_mat = ssm.T
Z_mat = ssm.Z
R_mat = ssm.R
Q_mat = ssm.Q
H_mat = ssm.H

fc_mean = np.zeros(n_forecast)
fc_var = np.zeros(n_forecast)

a = a_T.copy()
P = P_T.copy()

for h in range(n_forecast):
    a = T_mat @ a
    P = T_mat @ P @ T_mat.T + R_mat @ Q_mat @ R_mat.T
    fc_mean[h] = (Z_mat @ a)[0]
    fc_var[h] = (Z_mat @ P @ Z_mat.T + H_mat)[0, 0]

fc_se = np.sqrt(fc_var)

# Create forecast dates
last_date = dates.iloc[-1]
fc_dates = pd.date_range(start=last_date + pd.DateOffset(months=3), periods=n_forecast, freq='QS')

fig, ax = plt.subplots(figsize=(14, 6))

# Plot last 40 observations + forecast
n_show = 40
ax.plot(dates[-n_show:], y[-n_show:], 'k-', linewidth=1, label='Observed')
ax.plot(fc_dates, fc_mean, 'b-', linewidth=1.5, label='Forecast')
ax.fill_between(fc_dates, fc_mean - 1.96 * fc_se, fc_mean + 1.96 * fc_se,
                alpha=0.2, color='blue', label='95% CI')
ax.axvline(dates.iloc[-1], color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('Log GDP')
ax.set_title(f'UCM Forecast — {n_forecast} Quarters Ahead')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Forecast (log GDP):')
for h in range(n_forecast):
    print(f'  h={h+1}: {fc_mean[h]:.4f} ± {1.96*fc_se[h]:.4f} '
          f'(GDP index: {np.exp(fc_mean[h]):.1f})')

In [ ]:
# Comparison with statsmodels UnobservedComponents
sm_model = sm.tsa.UnobservedComponents(
    y,
    level='local linear trend',
    cycle=True,
    stochastic_cycle=True,
    damped_cycle=True,
    cycle_period_bounds=(16, 48),  # 4-12 years in quarters
)
sm_results = sm_model.fit(disp=False)

print('statsmodels UCM Results:')
print(sm_results.summary())

In [ ]:
# Comparison table: kalmanbox vs statsmodels
sm_freq = sm_results.params[sm_results.param_names.index('frequency.cycle')]
sm_rho = sm_results.params[sm_results.param_names.index('damping.cycle')]
sm_period = 2 * np.pi / sm_freq

comparison = pd.DataFrame({
    'Metric': [
        'Log-Likelihood',
        'Cycle Period (quarters)',
        'Cycle Period (years)',
        'Damping (ρ)',
        'σ²_level',
        'σ²_cycle',
    ],
    'kalmanbox': [
        f'{-final_result.fun:.2f}',
        f'{period_hat:.1f}',
        f'{period_hat/4:.1f}',
        f'{rho_hat:.4f}',
        f'{optimal_params[ucm.param_names.index("sigma2_level")]:.6f}',
        f'{optimal_params[ucm.param_names.index("sigma2_cycle")]:.6f}',
    ],
    'statsmodels': [
        f'{sm_results.llf:.2f}',
        f'{sm_period:.1f}',
        f'{sm_period/4:.1f}',
        f'{sm_rho:.4f}',
        f'{sm_results.params[sm_results.param_names.index("sigma2.level")]:.6f}',
        f'{sm_results.params[sm_results.param_names.index("sigma2.cycle")]:.6f}',
    ],
})

print('\nComparison: kalmanbox vs statsmodels')
print('=' * 70)
print(comparison.to_string(index=False))

# Plot smoothed components comparison
sm_level = sm_results.level['smoothed']
sm_cycle_component = sm_results.cycle['smoothed']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(dates, level, 'b-', linewidth=1.5, label='kalmanbox level')
axes[0].plot(dates, sm_level, 'r--', linewidth=1.5, label='statsmodels level', alpha=0.8)
axes[0].plot(dates, y, 'k-', linewidth=0.5, alpha=0.3, label='Observed')
axes[0].set_ylabel('Log GDP')
axes[0].set_title('Level Component: kalmanbox vs statsmodels')
axes[0].legend()

axes[1].plot(dates, cycle * 100, 'b-', linewidth=1.5, label='kalmanbox cycle')
axes[1].plot(dates, sm_cycle_component * 100, 'r--', linewidth=1.5, label='statsmodels cycle', alpha=0.8)
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Cycle (%)')
axes[1].set_title('Cycle Component: kalmanbox vs statsmodels')
axes[1].legend()

plt.tight_layout()
plt.show()

## Conclusions

1. **UCM Decomposition**: The UCM successfully decomposes Brazil's quarterly GDP
   into trend, cycle, and irregular components.

2. **Business Cycle**: The estimated cycle period is in the range of **4-12 years**,
   consistent with the typical duration of business cycles observed in emerging
   economies. The damping factor $\rho$ indicates how quickly cycle shocks dissipate.

3. **Trend-Cycle Identification**: Separating trend from cycle is a well-known
   identification challenge in structural time series (Harvey, 1989; Morley et al., 2003).
   Constraining the cycle frequency to the business-cycle range (4-12 years) is
   standard practice and improves identification.

4. **Comparison with statsmodels**: Both implementations recover similar decompositions.
   statsmodels uses explicit `cycle_period_bounds` to constrain the frequency;
   kalmanbox achieves the same via bounded optimization or profile likelihood.

### References
- Harvey, A.C. (1989). *Forecasting, Structural Time Series Models and the Kalman Filter*.
- Clark, P.K. (1987). The Cyclical Component of U.S. Economic Activity. *QJE*.
- Durbin, J. & Koopman, S.J. (2012). *Time Series Analysis by State Space Methods*.